**MIRAGE DATASET** (copy from MIRAGE repo) and gathered with Claude Sonnet

| Column | Type | Notes |
|---|---|---|
| `source` | str | `popqa` / `naturalqa` / `triviaqa` / `ifqa` / `drop` |
| `query_id` | str | UUID — also stored as `oracle['mapped_id']` |
| `query` | str | Question text |
| `doc_name` | str | Oracle Wikipedia article title |
| `answer` | list[str] | One or more gold answer strings |
| `doc_url` | str | Wikipedia URL |
| `num_doc_labels` | int | Number of supporting chunks |
| `oracle` | dict | `{mapped_id, doc_name, doc_chunk, support}` — oracle passage, one per row |
| `doc_pool` | dict of lists | 5 candidate chunks per row: parallel lists `mapped_id`, `doc_name`, `doc_chunk`, `support` |

### Технические ячейки

In [ ]:
!pip install -q datasets evaluate bert-score accelerate

Unknown instance spec: Please select VM configuration

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from bert_score import score as bert_score_fn
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM


import re
import ast
import string
import random
import warnings

from typing import List, Dict, Any

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 52
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
hf_dataset = load_dataset("nlpai-lab/mirage")["train"]
print(hf_dataset)
print(f"Column names: {hf_dataset.column_names}")

In [ ]:
dataset: List[Dict[str, Any]] = hf_dataset.to_list()

row = dataset[0]

print(row.keys())

In [ ]:
---
#### Подсчет основных метрик вынес в отдельную ячейку

**EM = exact matching**

| Metric | Normalisation | Rule |
|---|---|---|
| **EM-loose** | lowercase only | any gold answer is a *substring* of the prediction |
| **EM-strict** | lowercase only | any gold answer *exactly equals* the prediction |
| **F1 (MIRAGE)** | lowercase only | set-based token precision/recall harmonic mean |

In [ ]:
def normalize_squad(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def squad_em(prediction: str, golds: List[str]) -> float:
    np = normalize_squad(prediction)
    return float(any(normalize_squad(g) == np for g in golds))

def squad_f1(prediction: str, golds: List[str]) -> float:
    def _pair(pred, gold):
        pt = set(normalize_squad(pred).split())
        gt = set(normalize_squad(gold).split())
        tp = len(pt & gt)
        if not tp:
            return 0.0
        p, r = tp / len(pt), tp / len(gt)
        return 2 * p * r / (p + r)
    return max(_pair(prediction, g) for g in golds)

def em_loose(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_EM_loose() в файле оригинального репо LLM.py.
    """
    pred_lc = prediction.lower()
    return float(any(g.lower() in pred_lc for g in golds))

def em_strict(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_EM_strict() в файле оригинального репо LLM.py.
    """
    pred_lc = prediction.lower()
    return float(any(g.lower() == pred_lc for g in golds))

def mirage_f1(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_f1() в файле оригинального репо LLM.py.
    """
    pred_tokens = set(prediction.lower().split())
    gold_tokens = set(tok for g in golds for tok in g.lower().split())
    tp = len(pred_tokens & gold_tokens)
    if not tp:
        return 0.0
    p = tp / len(pred_tokens) if pred_tokens else 0.0
    r = tp / len(gold_tokens)  if gold_tokens  else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0

In [ ]:
def _to_gold_list(raw) -> List[str]:
    if isinstance(raw, list):
        return [str(v) for v in raw]
    if isinstance(raw, str):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return [str(v) for v in parsed]
        except Exception:
            pass
        return [raw]
    return [str(raw)]

def evaluate_answers(
    predictions: List[str],
    references: List[Any],
    bertscore_model: str = "roberta-large",
    bs_batch_size: int = 32,
) -> Dict[str, float]:

    em_s, f1_s, loose_s, strict_s, mf1_s = [], [], [], [], []

    for pred, raw in zip(predictions, references):
        golds = _to_gold_list(raw)
        em_s.append(squad_em(pred, golds))
        f1_s.append(squad_f1(pred, golds))
        loose_s.append(em_loose(pred, golds))
        strict_s.append(em_strict(pred, golds))
        mf1_s.append(mirage_f1(pred, golds))

    flat_refs = [_to_gold_list(r)[0] for r in references]
    _, _, bsf1 = bert_score_fn(
        predictions, flat_refs,
        model_type=bertscore_model,
        lang="en",
        batch_size=bs_batch_size,
        verbose=False,
        device=DEVICE,
    )

    def _pct(lst): return round(sum(lst) / len(lst), 3)

    return {
        "EM (SQuAD-like)": _pct(em_s),
        "F1 (SQuAD-like)": _pct(f1_s),
        "EM-loose (MIRAGE)": _pct(loose_s),
        "EM-strict (MIRAGE)": _pct(strict_s),
        "F1 (MIRAGE)": _pct(mf1_s),
        "BERTScore-F1": round(bsf1.mean().item(), 3),
    }

## Task 2
### 2.1 Extract questions, gold answers, and oracle passages

The oracle passage lives directly on each row at `row['oracle']['doc_chunk']`.
No secondary lookup is needed — this mirrors how `generate_LLM_prompt` uses
`self.oracle[data_dict['query_id']]['doc_chunk']` after building the dict from the same column.

In [ ]:
questions = [row["query"] for row in dataset]
gold_answers = [_to_gold_list(row["answer"]) for row in dataset]
oracle_ctxs = [row["oracle"]["doc_chunk"] for row in dataset]

print(f"Всего строк: {len(questions)}")

In [ ]:
print(f"{questions[0]}")

In [ ]:
print(f"Голден ответы: {gold_answers[0]}")

### Берем QA-пайплайн из библиотеки transformers


In [ ]:
MRC_MODEL = "deepset/roberta-base-squad2"

mrc_pipe = pipeline(
    "question-answering",
    model=MRC_MODEL,
    device=0 if DEVICE == "cuda" else -1,
)
print(f"Loaded: {MRC_MODEL}")

In [ ]:
from tqdm.notebook import tqdm
MRC_BATCH = 16

mrc_inputs = [
    {"question": q, "context": c}
    for q, c in tqdm(zip(questions, oracle_ctxs), desc="input creation")
]
mrc_outputs = mrc_pipe(mrc_inputs, batch_size=MRC_BATCH, truncation=True, max_seq_len=512)М

In [ ]:
task2_preds = [out["answer"] for out in tqdm(mrc_outputs)]

In [ ]:
print("Качество работы. Визуальная оценка\n")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"Prediction: {task2_preds[i]}")
    print(f"Голден: {gold_answers[i]}")

На первый взгляд, неплохо. В 3 из 4 случаев модель отвечает правильно. Что же будет по метрикам?